# 📘 Agentic 架构 9：思维树（Tree-of-Thoughts）规划

欢迎阅读本系列第九个 notebook。今天探讨一种强大的推理与规划架构：**思维树（Tree-of-Thoughts, ToT）**。它将智能体解题能力从线性思维链提升为多路径探索式搜索。

ToT 在问题的每一阶段生成多个候选「思维」或下一步，再评估这些思维，剪除无效或无望分支，扩展最有希望的分支，从而形成可回溯、可探索替代方案、系统化遍历复杂问题空间的搜索树。

为演示这一点，我们让智能体求解经典逻辑谜题：**狼、羊、白菜问题**。该谜题因需要非显然步骤（例如把某物 *带回*）以及多种会困住朴素推理者的非法状态而著名。我们将展示简单思维链（Chain-of-Thought）智能体可能失败，而 ToT 智能体如何通过探索与评估可能性树有条理地构造合法计划。

### 补充：先理解 CoT（Chain-of-Thought）

**思维链（CoT）** 是一种让模型把中间推理步骤按顺序显式写出来的方法。遇到复杂问题时，它不是直接跳到最终答案，而是先拆分任务，再沿着 **单一路径** 一步步推到结论。

可以把 CoT 理解为：**让模型把“脑内草稿”写出来**。这通常能提升多步推理题、数学题、逻辑题的稳定性，因为模型不再只做一次性猜测，而是先形成若干中间结论，再汇总成最终答案。

### CoT 的典型流程

1. **理解目标：** 明确问题、约束和期望输出。
2. **线性展开：** 依次写出当前判断、依据和下一步。
3. **累积中间结论：** 把前一步结果作为后一步输入。
4. **给出答案：** 在完整推理链末尾输出最终结论。

### CoT 的优势与局限

* **优势：** 实现简单、成本较低，适合一条推理路径就能解决的问题。
* **局限：** 它通常只会沿着一条思路往前走；如果前面某一步走偏，后面往往会“错上加错”，也不擅长显式比较多个候选方案。

这正是 **ToT（Tree-of-Thoughts）** 要扩展的地方：CoT 强在把一条路讲清楚，而 ToT 强在同时探索多条路、评估好坏并剪枝。

### CoT 代码举例

下面是一个最小化的 CoT 示例。它要求模型对狼、羊、白菜问题进行**逐步推理**，但整体仍然是一次性、单路径地完成求解。后文阶段 3 会运行一个同类 baseline，与 ToT 进行对比。

```python
from langchain_core.prompts import ChatPromptTemplate

problem = """
A farmer wants to cross a river with a wolf, a goat, and a cabbage.
The boat can only carry the farmer and one other item.
The farmer cannot leave the wolf alone with the goat,
nor the goat alone with the cabbage.
How can the farmer get everyone across safely?
"""

cot_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a world-class logic puzzle solver. Think step by step and provide one best solution path."
    ),
    ("human", "{problem}")
])

# llm 会在后文环境配置完成后初始化
cot_chain = cot_prompt | llm
cot_result = cot_chain.invoke({"problem": problem}).content
print(cot_result)
```

这个写法的关键特征是：模型会显式写出中间步骤，但**不会**主动维护多个候选分支，也**不会**对分支做系统性的评估与剪枝。后面的 ToT 实现正是为了解决这个问题。

### TOT定义
**思维树（ToT）**是一种将解题建模为树搜索的智能体推理框架。智能体同时探索多条推理路径（分支）。每一步生成可能的下一步（「思维」），评估其可行性，并决定继续扩展哪些路径，从而剪枝搜索空间。

### 高层工作流

1. **分解：**问题被拆为一系列步骤或思维。
2. **思维生成：**对问题当前状态，智能体生成多个可能的下一步或思维，在搜索树上形成分支。
3. **状态评估：**每个新思维（对应新状态）由「评判器」或验证函数评估，可包括：
    * **合法性：**该步是否符合问题规则？
    * **进展：**是否更接近解？
    * **启发式：**该路径是否可能成功？
4. **剪枝与扩展：**剪除非法或无望分支，再从最有希望的活跃分支继续重复思维生成。
5. **求解：**直到达到目标状态；解为从根到目标的思维路径。

### 适用场景 / 应用
* **逻辑谜题与数学问题：**规则与目标状态清晰、需要多步非线性推理的问题（如数独、过河谜题）。
* **复杂规划：**操作顺序重要且需满足约束的详细计划（如多段行程与预算约束的旅行规划）。
* **创意写作或代码生成：**在定稿前探索多个故事分支或实现策略。

### 优势与局限
* **优势：**
    * **稳健性：**系统化探索问题空间，相比单次方法更不易卡住或给出错误答案。
    * **应对组合复杂性：**适合可能序列数量巨大的问题。
* **局限：**
    * **计算成本高：**比简单思维链提示需要更多 LLM 调用与状态管理，更慢、更贵。
    * **依赖良好评估器：**搜索效果很大程度上取决于状态评估逻辑的质量。

## 阶段 0：基础与环境

照常安装库并配置 API 密钥。

In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv

In [2]:
import os
import re
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from collections import defaultdict

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.tree import Tree

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Tree-of-Thoughts (DeepSeek)"

# Check for required environment variables
required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：定义问题环境

思维树系统需要在明确定义的环境中运行。对狼、羊、白菜谜题，我们需要在程序中定义：

1. **状态表示：**描述各物体位置的方式。
2. **验证规则：**检查状态是否非法的函数（例如羊与白菜独处）。
3. **目标状态：**判断谜题是否已解。
4. **可行移动：**从给定状态枚举所有合法移动的函数。

In [3]:
console = Console()

class PuzzleState(BaseModel):
    "Represents the state of the Wolf, Goat, and Cabbage puzzle."
    # Using sets for unordered collections of items on each bank
    left_bank: set[str] = Field(default_factory=lambda: {"wolf", "goat", "cabbage"})
    right_bank: set[str] = Field(default_factory=set)
    boat_location: str = "left"
    move_description: str = "Initial state."

    def is_valid(self) -> bool:
        """Checks if the current state is valid (no one gets eaten)."""
        # Check left bank
        if self.boat_location == "right":
            if "wolf" in self.left_bank and "goat" in self.left_bank:
                return False
            if "goat" in self.left_bank and "cabbage" in self.left_bank:
                return False
        # Check right bank
        if self.boat_location == "left":
            if "wolf" in self.right_bank and "goat" in self.right_bank:
                return False
            if "goat" in self.right_bank and "cabbage" in self.right_bank:
                return False
        return True

    def is_goal(self) -> bool:
        """Checks if the goal state has been reached."""
        return self.right_bank == {"wolf", "goat", "cabbage"}
    
    def __hash__(self):
        # Make the state hashable to check for visited states
        return hash((frozenset(self.left_bank), frozenset(self.right_bank), self.boat_location))
    
    def __eq__(self, other):
        return self.__hash__() == other.__hash__()

def get_possible_moves(state: PuzzleState) -> list[PuzzleState]:
    """Generates all possible valid next states from the current state."""
    moves = []
    current_bank = state.left_bank if state.boat_location == "left" else state.right_bank
    
    # Option 1: Move one item in the boat
    for item in current_bank:
        new_state = state.model_copy(deep=True)
        if state.boat_location == "left":
            new_state.left_bank.remove(item)
            new_state.right_bank.add(item)
            new_state.boat_location = "right"
            new_state.move_description = f"Move {item} to the right bank."
        else:
            new_state.right_bank.remove(item)
            new_state.left_bank.add(item)
            new_state.boat_location = "left"
            new_state.move_description = f"Move {item} to the left bank."
        if new_state.is_valid():
            moves.append(new_state)
            
    # Option 2: Move the boat empty
    empty_move_state = state.model_copy(deep=True)
    if state.boat_location == "left":
        empty_move_state.boat_location = "right"
        empty_move_state.move_description = "Move the boat empty to the right bank."
    else:
        empty_move_state.boat_location = "left"
        empty_move_state.move_description = "Move the boat empty to the left bank."
    if empty_move_state.is_valid():
        moves.append(empty_move_state)
        
    return moves

print("Puzzle environment defined successfully.")

Puzzle environment defined successfully.


## 阶段 2：用 LangGraph 实现 ToT 智能体

接下来构建智能体本身。图状态跟踪思维树中所有活跃路径（分支）。节点执行 ToT 关键动作：

1. **扩展路径（思维生成器）：**由 LLM 驱动的节点，查看每条活跃路径的最后状态，并从合法可能性中提出有希望的下一步。
2. **剪枝路径（状态评估器）：**在生成后清理，移除进入非法状态或循环（重复先前状态）的路径。
3. **检查解（目标检测）：**条件节点，检查是否有路径到达目标；若有则终止循环。

In [4]:
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0.4,
)

# Pydantic model for the LLM's choice of move
class MoveChoice(BaseModel):
    best_move_index: int = Field(description="The index of the best move from the provided list of possible moves.")
    reasoning: str = Field(description="Brief reasoning for why this is the most promising move.")

# LangGraph State
class ToTState(TypedDict):
    problem_description: str
    # Each path is a list of PuzzleState objects
    active_paths: List[List[PuzzleState]]
    # We'll store the final solution here
    solution: Optional[List[PuzzleState]]

# Graph Nodes
def initialize_search(state: ToTState) -> Dict[str, Any]:
    """Node to set up the initial state of the search."""
    initial_puzzle_state = PuzzleState()
    return {"active_paths": [[initial_puzzle_state]]}

def expand_paths(state: ToTState) -> Dict[str, Any]:
    """The 'Thought Generator'. Expands each active path with a promising next move."""
    console.print("--- Expanding Paths ---")
    new_paths = []
    choice_llm = llm.with_structured_output(MoveChoice)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert logic puzzle solver. Your goal is to solve the Wolf, Goat, and Cabbage problem. Analyze the current path and choose the most promising next move from the list of options to reach the goal."),
        ("human", "Problem: {problem}\n\nCurrent Path History:\n{path_history}\n\nFrom the final state, choose the best next move from this list:\n{possible_moves}")
    ])
    
    for path in state['active_paths']:
        last_state = path[-1]
        possible_next_states = get_possible_moves(last_state)
        
        if not possible_next_states:
            continue # This path is a dead end
            
        path_history_str = " -> ".join([s.move_description for s in path])
        possible_moves_str = "\n".join([f"{i}: {s.move_description}" for i, s in enumerate(possible_next_states)])
        
        # For simplicity and to show breadth, we can explore multiple moves.
        # A more advanced ToT might use the LLM to pick only the single best one.
        # Here, we'll let all valid moves branch out to demonstrate the tree structure.
        for next_state in possible_next_states:
            new_paths.append(path + [next_state])

    console.print(f"[cyan]Expanded to {len(new_paths)} potential paths.[/cyan]")
    return {"active_paths": new_paths}

def prune_paths(state: ToTState) -> Dict[str, Any]:
    """The 'State Evaluator'. Prunes paths that are invalid or contain cycles."""
    console.print("--- Pruning Paths ---")
    pruned_paths = []
    for path in state['active_paths']:
        # Check for cycles: if the last state has appeared before in the path
        if path[-1] in path[:-1]:
            continue # Found a cycle, prune this path
        
        # The get_possible_moves function already ensures validity, but this is a good place for extra checks.
        pruned_paths.append(path)
        
    console.print(f"[green]Pruned down to {len(pruned_paths)} valid, non-cyclical paths.[/green]")
    return {"active_paths": pruned_paths}

# Conditional Node
def check_for_solution(state: ToTState) -> str:
    """Checks if any path has reached the goal and routes execution."""
    for path in state['active_paths']:
        if path[-1].is_goal():
            console.print("[bold green]Solution Found![/bold green]")
            # Side effect: update the solution in the state. LangGraph copies this out.
            state['solution'] = path
            return "solution_found"
    return "continue_search"

# Build the graph
workflow = StateGraph(ToTState)

workflow.add_node("initialize", initialize_search)
workflow.add_node("expand", expand_paths)
workflow.add_node("prune", prune_paths)

workflow.set_entry_point("initialize")
workflow.add_edge("initialize", "expand")
workflow.add_edge("expand", "prune")

workflow.add_conditional_edges(
    "prune",
    check_for_solution,
    {
        "solution_found": END,
        "continue_search": "expand"
    }
)

tot_agent = workflow.compile()
print("Tree-of-Thoughts agent graph compiled successfully.")

Tree-of-Thoughts agent graph compiled successfully.


## 阶段 3：演示与分析

在谜题上运行 ToT 智能体。将其系统化方法与单次思维链请求对比，突出稳健性差异。

In [5]:
problem = "A farmer wants to cross a river with a wolf, a goat, and a cabbage. The boat can only carry the farmer and one other item. The farmer cannot leave the wolf alone with the goat, nor the goat alone with the cabbage. How can the farmer get everyone across safely?"

console.print("--- 🌳 Running Tree-of-Thoughts Agent ---")
# The recursion limit prevents infinite loops in our graph
config = {"recursion_limit": 15}
final_state = tot_agent.invoke({"problem_description": problem}, config=config)

console.print("\n--- ✅ ToT Agent Solution ---")
if final_state.get('solution'):
    solution_path = final_state['solution']
    # Use rich.Tree for a nice visual output
    tree = Tree("[bold magenta]Wolf, Goat, and Cabbage Solution Path[/bold magenta]")
    for i, state in enumerate(solution_path):
        tree.add(f"[green]{i+1}.[/green] {state.move_description}")
    console.print(tree)
else:
    console.print("[bold red]No solution found within the step limit.[/bold red]")

console.print("\n--- 🤔 Running Simple Chain-of-Thought Agent ---")
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class logic puzzle solver. Provide a step-by-step solution to the user's puzzle."),
    ("human", "{problem}")
])
cot_chain = cot_prompt | llm
cot_result = cot_chain.invoke({"problem": problem}).content
console.print(Markdown(cot_result))

--- 🌳 Running Tree-of-Thoughts Agent ---


--- Expanding Paths ---
Expanded to 1 potential paths.
--- Pruning Paths ---
Pruned down to 1 valid, non-cyclical paths.
--- Expanding Paths ---
Expanded to 2 potential paths.
--- Pruning Paths ---
Pruned down to 2 valid, non-cyclical paths.
--- Expanding Paths ---
Expanded to 4 potential paths.
--- Pruning Paths ---
Pruned down to 4 valid, non-cyclical paths.
--- Expanding Paths ---
Expanded to 7 potential paths.
--- Pruning Paths ---
Pruned down to 7 valid, non-cyclical paths.
--- Expanding Paths ---
Expanded to 12 potential paths.
--- Pruning Paths ---
Pruned down to 12 valid, non-cyclical paths.
--- Expanding Paths ---
Expanded to 20 potential paths.
--- Pruning Paths ---
Pruned down to 20 valid, non-cyclical paths.
--- Expanding Paths ---
Expanded to 32 potential paths.
--- Pruning Paths ---
Pruned down to 32 valid, non-cyclical paths.
Solution Found!



--- ✅ ToT Agent Solution ---


Wolf, Goat, and Cabbage Solution Path
├── 1. Initial state.
├── 2. Move goat to the right bank.
├── 3. Move the boat empty to the left bank.
├── 4. Move wolf to the right bank.
├── 5. Move goat to the left bank.
├── 6. Move cabbage to the right bank.
├── 7. Move the boat empty to the left bank.
└── 8. Move goat to the right bank.



--- 🤔 Running Simple Chain-of-Thought Agent ---


Here's a step-by-step solution to the Wolf, Goat, and Cabbage puzzle:

1.  **Take the Goat across:** First, take the goat across the river to the right bank. You leave the wolf and cabbage behind on the left bank.
2.  **Return empty:** Return to the left bank alone.
3.  **Take the Wolf across:** Now, take the wolf across to the right bank. 
4.  **Bring the Goat back:** *This is the key step.* Leave the wolf on the right bank and bring the goat back with you to the left bank.
5.  **Take the Cabbage across:** Leave the goat on the left bank and take the cabbage across to the right bank. Now the wolf and cabbage are on the right bank.
6.  **Return empty:** Return to the left bank alone.
7.  **Take the Goat across:** Finally, take the goat across to the right bank.

Now, the wolf, goat, and cabbage are all safely on the right bank, and the puzzle is solved.

### 结果分析

两种方法差异显著：

- **思维链（CoT）：**依赖 LLM 预训练知识回忆解法。对这类经典、广为人知的问题，强模型常能一次答对。但若犯一个错误，没有自我纠正机制；对新颖或更复杂问题，失败概率高得多。正确性来自回忆，而非可验证推理。

- **思维树（ToT）：**智能体通过系统化、可验证的 **搜索** *发现* 解，而非仅回忆答案。从日志可见：扩展路径，再剪除进入死路或循环的分支。即便扩展引导的 LLM 在某一分支上选择欠佳，仍可继续探索其他更有希望的分支。该方法更稳健、可信，因为最终解在我们定义的环境规则下保证合法。

ToT 的成功不依赖运气或死记硬背，而依赖搜索算法的正确性，因此在要求高可靠性与规划的任务上远为优越。

## 结语

本 notebook 实现了 **思维树** 智能体求解经典逻辑谜题。通过将问题转化为状态空间并系统化搜索，智能体可达到简单单次推理无法实现的稳健性与准确度。

ToT 的核心组件——**思维生成（扩展）**、**状态评估（剪枝）** 与 **搜索**——为复杂规划与推理任务提供了有力框架。计算成本更高，但换来可靠性与解题能力的显著提升。该架构是构建能够审慎推理、求解富有挑战的多步问题的智能体的关键一步。